In [0]:
Gen AI: simple app. 
(rag)
Agent: do action 
agentic ai: gmail (mcp)

agent: 
    LLM (brain) + Hands(Tools) 

In [0]:
%pip install -U --quiet \
    databricks-langchain \
    langchain \
    langchain-community \
    wikipedia \
    youtube_search \
    duckduckgo-search\
    -U ddgs \
    -U langgraph

dbutils.library.restartPython()

In [0]:
import mlflow

# Set experiment for better organization
mlflow.set_experiment("/Users/naval.datamaster@gmail.com/agent")

# Enable autologging BEFORE creating the LangChain client
mlflow.langchain.autolog()

In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import YouTubeSearchTool
from langchain_community.tools import DuckDuckGoSearchRun

In [0]:
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

result = wiki_tool.invoke("Generative AI?")

print(result)

In [0]:
youtube_tool = YouTubeSearchTool()

result = youtube_tool.invoke("Independence day 2026")

print(result)

In [0]:
search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke("Latest Databricks AI features")

print(result)

In [0]:
tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]

for tool in tools:
    print("Name:", tool.name)
    print("Description:", tool.description)
    print("-" * 50)

In [0]:
User
 ↓
LLM
 ↓
Decide which tool to use
 ↓
Tool
 ↓
Tool result
 ↓
LLM
 ↓
Final answer

In [0]:
User
 ↓
Movie Agent
 ↓
Wikipedia → movie information
 ↓
YouTube → trailer
 ↓
Web Search → latest information
 ↓
LLM
 ↓
Recommendation

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# LLM

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]

# Agent
agent = create_agent(
    model=llm,
    tools=tools
)


# User query
query = """
Recommend a good science-fiction movie.

Use Wikipedia to find information about the movie
and YouTube to find its trailer.

Give me:
1. Movie name
2. Release year
3. Short description
4. Why you recommend it
5. YouTube trailer
"""


# Invoke agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})


# Final answer
print(result["messages"][-1].content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# -----------------------------------
# 1. Databricks Foundation Model
# -----------------------------------

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)


# -----------------------------------
# 2. Tools
# -----------------------------------

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]


# -----------------------------------
# 3. Create Agent
# -----------------------------------

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful movie research assistant.

Use:
- Wikipedia for factual movie information
- YouTube for movie trailers
- Web search for current information

Always use the appropriate tools to verify information.
"""
)


# -----------------------------------
# 4. Get Movie Name from User
# -----------------------------------

movie_name = input("🎬 Enter the movie name: ")

query = f"""
Tell me about the movie "{movie_name}".

Use Wikipedia to find information about the movie
and YouTube to find its trailer.

Provide:

1. Movie name
2. Release year
3. Director
4. Main actors
5. Short description
6. Why someone should watch it
7. YouTube trailer
"""


# -----------------------------------
# 5. Invoke Agent
# -----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})


# -----------------------------------
# 6. Display Final Answer
# -----------------------------------

print("\n🎬 Movie Information")
print("=" * 60)

print(result["messages"][-1].content)